# PointNet: 3D Room Semantic Segmentation

## About this activity
Use a pretrained PointNet model to classify points in a 3D room.

You will:
1. Download the classroom materials.
2. Convert raw annotations into a point cloud.
3. Explore the original room in interactive 3D.
4. Load the pretrained model.
5. Run semantic segmentation and calculate point accuracy.
6. Compare predictions with ground-truth labels.

This activity performs inference using a pretrained model.
It does not train a new model.

In [ ]:
%pip install -q gdown plotly tqdm scipy

from pathlib import Path
import zipfile
import gdown

# Download the teacher-provided classroom package.
file_id = "1dTlVr1zrm2-8OixYTQIDMvBeBckBKoEZ"

work_dir = Path("/content/pointnet_classroom")
work_dir.mkdir(parents=True, exist_ok=True)

zip_path = work_dir / "pointnet_classroom.zip"

print("Downloading classroom materials...", flush=True)

result = gdown.download(
    id=file_id,
    output=str(zip_path),
    quiet=False,
    use_cookies=False
)

assert result is not None, "Download failed. Ask your teacher for help."
assert zipfile.is_zipfile(zip_path), "The downloaded file is not a valid ZIP."

# Validate archive paths before extraction.
print("Extracting classroom materials...", flush=True)

with zipfile.ZipFile(zip_path, "r") as archive:
    for member in archive.infolist():
        destination = (work_dir / member.filename).resolve()
        assert destination.is_relative_to(work_dir.resolve()), (
            "An unsafe archive path was detected."
        )

    archive.extractall(work_dir)

# Define paths for the following notebook cells.
project = work_dir / "Pointnet_Pointnet2_pytorch"
annotation_dir = (
    work_dir / "room_source/Area_5/office_1/Annotations"
)
checkpoint_path = (
    project
    / "log/sem_seg/pointnet_sem_seg/checkpoints/best_model.pth"
)

# Check the required materials.
required_files = [
    project / "models/pointnet_sem_seg.py",
    project / "models/pointnet_utils.py",
    project / "data_utils/S3DISDataLoader.py",
    checkpoint_path,
]

for path in required_files:
    assert path.is_file(), f"Required file missing: {path.name}"

annotation_files = sorted(annotation_dir.glob("*.txt"))
assert len(annotation_files) == 39, (
    f"Expected 39 annotation files, found {len(annotation_files)}."
)

print("\nStep 1 completed successfully!")
print(f"Original annotation files: {len(annotation_files)}")
print("Model code: READY")
print("Pretrained weights: READY")
print("Next step: Convert the raw room annotations.")

## Step 2: Convert the raw room annotations

The room contains 39 object annotation files.

In this step, you will:
1. Read each object's point coordinates and RGB colors.
2. Assign its ground-truth class label using the file name.
3. Combine all objects into one room.
4. Shift the coordinates so that the minimum of each axis is zero.
5. Save the result as a NumPy file.

Each row in the output represents one point:
X, Y, Z, R, G, B, Label.

These labels are human annotations, not model predictions.
No model training or inference is performed in this step.

In [ ]:
from pathlib import Path
import numpy as np

# Locate the materials downloaded in Step 1.
work_dir = Path("/content/pointnet_classroom")
project = work_dir / "Pointnet_Pointnet2_pytorch"
annotation_dir = (
    work_dir / "room_source/Area_5/office_1/Annotations"
)

# Use the class order expected by the pretrained model.
class_names = [
    "ceiling", "floor", "wall", "beam", "column",
    "window", "door", "table", "chair", "sofa",
    "bookcase", "board", "clutter"
]

class_to_label = {
    name: index for index, name in enumerate(class_names)
}

annotation_files = sorted(annotation_dir.glob("*.txt"))

assert len(annotation_files) == 39, (
    "Expected 39 annotation files. Complete Step 1 first."
)

print("Reading raw annotations...", flush=True)

room_parts = []

for index, path in enumerate(annotation_files, start=1):
    # Obtain the object class from the file name.
    class_name = path.stem.split("_")[0]

    if class_name not in class_to_label:
        print(f"Unknown class mapped to clutter: {class_name}")
        class_name = "clutter"

    # Read the six columns: X, Y, Z, R, G, B.
    points = np.loadtxt(path, dtype=np.float64, ndmin=2)

    assert points.shape[0] > 0, f"Empty file: {path.name}"
    assert points.shape[1] == 6, (
        f"Expected six columns in: {path.name}"
    )
    assert np.isfinite(points).all(), (
        f"Invalid numeric values in: {path.name}"
    )

    # Add the ground-truth class label as the seventh column.
    labels = np.full(
        (len(points), 1),
        class_to_label[class_name],
        dtype=np.float64
    )

    room_parts.append(np.concatenate([points, labels], axis=1))

    print(
        f"[{index:02d}/{len(annotation_files)}] "
        f"{path.name}: {len(points):,} points"
    )

# Merge all objects into one room.
room = np.concatenate(room_parts, axis=0)

# Shift coordinates while preserving the room's geometry.
room[:, :3] -= room[:, :3].min(axis=0)

assert ((room[:, 3:6] >= 0) & (room[:, 3:6] <= 255)).all(), (
    "RGB values must be between 0 and 255."
)

# Save student-generated data separately from the backup file.
data_root = work_dir / "student_data"
data_root.mkdir(parents=True, exist_ok=True)

room_file = data_root / "Area_5_office_1.npy"
np.save(room_file, room)

# Verify that the saved file matches the generated data.
saved_room = np.load(room_file, allow_pickle=False)
assert np.array_equal(saved_room, room), (
    "Saved file verification failed."
)

print("\nStep 2 completed successfully!")
print(f"Total points: {len(room):,}")
print(f"Array shape: {room.shape}")
print("Columns: X, Y, Z, R, G, B, Label")
print(f"Saved to: {room_file}")
print("Next step: Explore the original room in interactive 3D.")

## Step 3: Explore the room in interactive 3D

Before running PointNet, explore the original room.

- Drag with the left mouse button to rotate.
- Scroll to zoom in or out.
- Identify the floor, walls, tables, and chairs.

For display only, we randomly sample up to 30,000 points.
We hide points labeled as ceiling to make the interior easier to see.

This visualization uses the original RGB colors, not model predictions.
The complete saved room data remains unchanged.

In [ ]:
from pathlib import Path
import numpy as np
import plotly.graph_objects as go

# Display settings.
MAX_POINTS = 30000
HIDE_CEILING = True
POINT_SIZE = 2

# Read the room generated by the student in Step 2.
room_file = Path(
    "/content/pointnet_classroom/student_data/"
    "Area_5_office_1.npy"
)

assert room_file.is_file(), "Complete Step 2 before running this cell."

room_data = np.load(room_file, allow_pickle=False)

assert room_data.ndim == 2 and room_data.shape[1] == 7, (
    "Expected a room array with seven columns."
)

# Select visible points without changing the saved data.
visible_mask = np.ones(len(room_data), dtype=bool)

if HIDE_CEILING:
    # Class 0 is the ground-truth ceiling label.
    visible_mask &= (room_data[:, 6] != 0)

available_indices = np.flatnonzero(visible_mask)
assert len(available_indices) > 0, "No points are available for display."

# Sample points only for visualization.
rng = np.random.default_rng(42)
display_indices = rng.choice(
    available_indices,
    size=min(MAX_POINTS, len(available_indices)),
    replace=False
)

display_points = room_data[display_indices]
xyz = display_points[:, :3]
rgb = np.clip(display_points[:, 3:6], 0, 255).astype(np.uint8)

rgb_colors = [
    f"rgb({int(r)},{int(g)},{int(b)})"
    for r, g, b in rgb
]

# Create an interactive view using the original RGB colors.
fig = go.Figure(
    go.Scatter3d(
        x=xyz[:, 0],
        y=xyz[:, 1],
        z=xyz[:, 2],
        mode="markers",
        marker=dict(
            size=POINT_SIZE,
            color=rgb_colors,
            opacity=1
        ),
        hovertemplate=(
            "X: %{x:.2f}<br>"
            "Y: %{y:.2f}<br>"
            "Z: %{z:.2f}"
            "<extra>Original point</extra>"
        )
    )
)

ceiling_status = "hidden" if HIDE_CEILING else "visible"

fig.update_layout(
    title=(
        "Area_5 office_1 | Original RGB"
        f" | Ceiling {ceiling_status}"
    ),
    height=700,
    margin=dict(l=0, r=0, b=0, t=60),
    scene=dict(
        aspectmode="data",
        dragmode="orbit",
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="Z",
        camera=dict(eye=dict(x=1.5, y=1.5, z=1.2))
    ),
    showlegend=False
)

fig.show(
    renderer="colab",
    config={
        "scrollZoom": True,
        "displaylogo": False
    }
)

print("\nStep 3 completed successfully!")
print(f"Full room: {len(room_data):,} points")
print(f"Displayed sample: {len(display_indices):,} points")
print(f"Ceiling: {ceiling_status}")
print("Colors: Original RGB")
print("The saved room data was not modified.")
print("No model predictions have been generated yet.")

## Step 4: Load the pretrained PointNet model

In this step, you will:
1. Create the PointNet segmentation model.
2. Prepare the model for inference.

The model predicts one of 13 semantic classes for each point.

Loading pretrained weights is not training.
We will run predictions on the room in the next step.

Security note: Only load checkpoint files from a trusted source.

In [ ]:
from pathlib import Path
import sys
import torch

# Locate the classroom project downloaded in Step 1.
work_dir = Path("/content/pointnet_classroom")
project = work_dir / "Pointnet_Pointnet2_pytorch"

checkpoint_path = (
    project
    / "log/sem_seg/pointnet_sem_seg/checkpoints/best_model.pth"
)

assert checkpoint_path.is_file(), (
    "Checkpoint not found. Complete Step 1 first."
)
assert (project / "models/pointnet_sem_seg.py").is_file(), (
    "Model code not found. Complete Step 1 first."
)

# Make the project modules available for import.
for folder in [project, project / "models"]:
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))

from pointnet_sem_seg import get_model

# Select an available device.
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"Selected device: {device}")

if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Using CPU. Full-room inference may take longer.")

print("Loading pretrained weights...", flush=True)

# Only load this checkpoint if you trust the classroom package.
checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

assert isinstance(checkpoint, dict), (
    "Unexpected checkpoint format."
)
assert "model_state_dict" in checkpoint, (
    "Model weights were not found in the checkpoint."
)

# Create the model and require an exact match of weight keys.
NUM_CLASSES = 13
model = get_model(NUM_CLASSES)
model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

# Move the model to the selected device.
model = model.to(device)
model.eval()

del checkpoint

print("\nStep 4 completed successfully!")
print(f"Device: {device}")
print(f"Number of classes: {NUM_CLASSES}")
print("Pretrained weights: LOADED")
print("Model: READY")
print("No training was performed.")
print("No room predictions have been generated yet.")
print("Next step: Run semantic segmentation on the room.")

## Step 5: Run semantic segmentation

Now use PointNet to predict a semantic class for every point in the room.

The program will:
1. Divide the full room into overlapping blocks of 4,096 points.
2. Run the pretrained model on each block.
3. Combine predictions using majority voting.
4. Compare the predictions with ground-truth labels.
5. Save the results for interactive visualization.

Ground-truth labels are not passed into the model.
The entire room is used, including the ceiling.

This activity runs one voting pass and does not train the model.

Point accuracy is the percentage of evaluated points whose predicted
class matches the ground-truth class. It is not mIoU or a full-dataset score.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import torch
from tqdm.auto import tqdm

# Locate the project and the student-generated room.
work_dir = Path("/content/pointnet_classroom")
project = work_dir / "Pointnet_Pointnet2_pytorch"
data_root = work_dir / "student_data"
room_name = "Area_5_office_1.npy"

assert (data_root / room_name).is_file(), (
    "Room data not found. Complete Step 2 first."
)
assert "model" in globals() and "device" in globals(), (
    "Complete Step 4 before running this cell."
)

if str(project) not in sys.path:
    sys.path.insert(0, str(project))

from data_utils.S3DISDataLoader import ScannetDatasetWholeScene

# Set random seeds for block sampling.
np.random.seed(42)
torch.manual_seed(42)

print(f"Device: {device}")
print("Loading the student-generated room...", flush=True)

# Some classes are absent from this single room.
# Class-frequency weights are not needed for this inference activity.
with np.errstate(divide="ignore", invalid="ignore"):
    dataset = ScannetDatasetWholeScene(
        root=str(data_root) + "/",
        split="test",
        test_area=5,
        block_points=4096
    )

dataset.labelweights = np.ones(13, dtype=np.float32)

assert room_name in dataset.file_list, "The target room was not found."
room_index = dataset.file_list.index(room_name)

print("Preparing room blocks. Please wait...", flush=True)

blocks, _, _, point_indices = dataset[room_index]

assert len(blocks) > 0, "No room blocks were generated."
assert np.isfinite(blocks).all(), (
    "The model input contains invalid numeric values."
)

xyzrgb = dataset.scene_points_list[room_index]
ground_truth = dataset.semantic_labels_list[room_index].astype(
    np.int64
)

print(f"Room: {room_name}")
print(f"Original points: {len(ground_truth):,}")
print(f"Blocks: {len(blocks):,}")
print("Points per block: 4096")

# Count the votes for each original point and each class.
votes = np.zeros((len(ground_truth), 13), dtype=np.int32)
batch_size = 4

model.eval()

# Only point features are passed into the model, not class labels.
with torch.inference_mode():
    for start in tqdm(
        range(0, len(blocks), batch_size),
        desc="Predicting"
    ):
        end = min(start + batch_size, len(blocks))

        inputs = torch.from_numpy(
            blocks[start:end].astype(np.float32)
        ).transpose(1, 2).contiguous().to(device)

        scores, _ = model(inputs)
        batch_predictions = scores.argmax(dim=2).cpu().numpy()

        original_indices = point_indices[start:end].astype(np.int64)

        np.add.at(
            votes,
            (original_indices.ravel(), batch_predictions.ravel()),
            1
        )

# Mark points without votes as unpredicted.
covered = votes.sum(axis=1) > 0
assert covered.any(), "No points received predictions."

prediction = np.full(len(ground_truth), -1, dtype=np.int64)
prediction[covered] = votes[covered].argmax(axis=1)

# Evaluate only points that received predictions.
accuracy = np.mean(
    prediction[covered] == ground_truth[covered]
)

# Save the full-room results for Step 6.
result_dir = work_dir / "results"
result_dir.mkdir(parents=True, exist_ok=True)

result_file = result_dir / "Area_5_office_1_prediction.npz"

np.savez_compressed(
    result_file,
    xyzrgb=xyzrgb,
    ground_truth=ground_truth,
    prediction=prediction,
    covered=covered
)

# Release large temporary arrays after saving the results.
del blocks, point_indices, votes

print("\nStep 5 completed successfully!")
print(f"Prediction coverage: {covered.mean():.2%}")
print(f"Point accuracy on covered points: {accuracy:.2%}")
print(f"Results saved to: {result_file}")
print("No training was performed.")
print("The pretrained checkpoint was not modified.")
print("Next step: Compare ground truth and predictions in interactive 3D.")

## Step 6: Compare the results in interactive 3D

Use the buttons above the viewer to switch between:

- Original RGB: the original point colors.
- Ground truth: the human-annotated classes.
- Prediction: the classes predicted by PointNet.
- Errors: red indicates an incorrect prediction; gray indicates a correct prediction.

Drag to rotate and scroll to zoom.
Hover over a point to see its ground-truth and predicted classes.

All modes display exactly the same sampled points.
For easier viewing, ground-truth ceiling points are hidden by default.

### Observation tasks
1. Record your full-room point accuracy.
2. Find one correctly classified object.
3. Find one misclassified region.
4. Hover over that region and record its true and predicted classes.

In [ ]:
from pathlib import Path
import numpy as np
import plotly.graph_objects as go

# Display settings.
MAX_POINTS = 30000
HIDE_CEILING = True
POINT_SIZE = 2

# Read the predictions generated in Step 5.
result_file = Path(
    "/content/pointnet_classroom/results/"
    "Area_5_office_1_prediction.npz"
)

assert result_file.is_file(), "Complete Step 5 first."

with np.load(result_file, allow_pickle=False) as result:
    xyzrgb = result["xyzrgb"]
    truth = result["ground_truth"].astype(np.int64)
    prediction = result["prediction"].astype(np.int64)
    covered = result["covered"].astype(bool)

class_names = np.array([
    "ceiling", "floor", "wall", "beam", "column",
    "window", "door", "table", "chair", "sofa",
    "bookcase", "board", "clutter"
])

# Use identical class colors for ground truth and predictions.
palette = np.array([
    "#00ff00", "#0000ff", "#00ffff", "#ffff00",
    "#ff00ff", "#6464ff", "#c8c864", "#aa78c8",
    "#ff0000", "#c86464", "#0ac864", "#c8c8c8",
    "#323232"
])

# Choose one shared set of points for all display modes.
visible = covered.copy()

if HIDE_CEILING:
    visible &= (truth != 0)

available = np.flatnonzero(visible)
assert len(available) > 0, "No points are available for display."

rng = np.random.default_rng(42)
indices = rng.choice(
    available,
    size=min(MAX_POINTS, len(available)),
    replace=False
)

xyz = xyzrgb[indices, :3]
rgb = np.clip(xyzrgb[indices, 3:6], 0, 255).astype(int)
sample_truth = truth[indices]
sample_prediction = prediction[indices]

rgb_colors = [
    f"rgb({r},{g},{b})" for r, g, b in rgb
]
truth_colors = palette[sample_truth].tolist()
prediction_colors = palette[sample_prediction].tolist()

correct = sample_truth == sample_prediction
error_colors = np.where(
    correct, "#b0b0b0", "#ff0000"
).tolist()

hover_text = [
    f"Ground truth: {class_names[t]}<br>"
    f"Prediction: {class_names[p]}<br>"
    f"Result: {'Correct' if t == p else 'Incorrect'}"
    for t, p in zip(sample_truth, sample_prediction)
]

# Create one point-cloud trace.
fig = go.Figure(go.Scatter3d(
    x=xyz[:, 0],
    y=xyz[:, 1],
    z=xyz[:, 2],
    mode="markers",
    marker=dict(
        size=POINT_SIZE,
        color=rgb_colors,
        opacity=1
    ),
    text=hover_text,
    hovertemplate="%{text}<extra></extra>",
    showlegend=False
))

# Change only the point colors when a button is clicked.
buttons = []

for label, colors in [
    ("Original RGB", rgb_colors),
    ("Ground truth", truth_colors),
    ("Prediction", prediction_colors),
    ("Errors", error_colors)
]:
    buttons.append(dict(
        label=label,
        method="restyle",
        args=[{"marker.color": [colors]}, [0]]
    ))

fig.update_layout(
    title="PointNet Results | Area_5 office_1",
    height=750,
    margin=dict(l=0, r=0, b=0, t=120),
    scene=dict(
        aspectmode="data",
        dragmode="orbit",
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="Z",
        camera=dict(eye=dict(x=1.5, y=1.5, z=1.2))
    ),
    updatemenus=[dict(
        type="buttons",
        direction="right",
        x=0,
        xanchor="left",
        y=1.10,
        yanchor="bottom",
        active=0,
        buttons=buttons
    )],
    uirevision="keep-camera"
)

fig.show(
    renderer="colab",
    config={"scrollZoom": True, "displaylogo": False}
)

# Report accuracy on all covered points, not the display sample.
full_accuracy = np.mean(
    prediction[covered] == truth[covered]
)

print("\nStep 6 completed successfully!")
print(f"Displayed points: {len(indices):,}")
print(f"Ground-truth ceiling hidden: {HIDE_CEILING}")
print(f"Full-room prediction coverage: {covered.mean():.2%}")
print(f"Point accuracy on all covered points: {full_accuracy:.2%}")
print("Errors mode: red = incorrect, gray = correct.")
print("Hover over a point to inspect its class labels.")
print("The saved data and predictions were not modified.")

## Step 7: Experiment with segmentation parameters

Modify these parameters to observe how they affect PointNet’s predictions:

- **BLOCK_SIZE:** the XY side length of each spatial block. Try `1.0`, `1.5`, or `2.0`.
- **STRIDE:** the distance between neighboring block positions. Try `0.5`, `0.75`, or `1.0`. Smaller values usually create more overlap and take longer to process.

Use `0 < STRIDE <= BLOCK_SIZE` and change only one parameter at a time.

Run the cell to report:

- **Overall point accuracy:** the percentage of evaluated points classified correctly.
- **Per-class IoU:** segmentation performance for each semantic class.
- **Room mIoU:** the average of valid class IoUs.

### Observation tasks

1. Record the point accuracy and room mIoU with the initial settings.
2. Change `BLOCK_SIZE`, keeping `STRIDE` fixed, and compare the results.
3. Change `STRIDE`, keeping `BLOCK_SIZE` fixed, and compare again.
4. Identify one class whose IoU improved or decreased.
5. Discuss whether the changes benefited all classes equally.

In [ ]:
from pathlib import Path
import sys
import csv
import numpy as np
import torch
from tqdm.auto import tqdm
# ============================================================
# Student settings.
BLOCK_SIZE = 1.5   # Try: 1.0, 1.5, 2.0
STRIDE = 1.0       # Try: 0.5, 0.75, 1.0
# ============================================================
# Fixed inference settings.
POINTS_PER_BLOCK = 4096
BATCH_SIZE = 4
NUM_CLASSES = 13
SEED = 42

CLASS_NAMES = [
    "ceiling", "floor", "wall", "beam", "column",
    "window", "door", "table", "chair", "sofa",
    "bookcase", "board", "clutter"
]

# ------------------------------------------------------------
# 1. Validate settings and locate the project.
# ------------------------------------------------------------
assert np.isfinite(BLOCK_SIZE) and BLOCK_SIZE > 0, (
    "BLOCK_SIZE must be a positive number."
)
assert np.isfinite(STRIDE) and 0 < STRIDE <= BLOCK_SIZE, (
    "STRIDE must be positive and no larger than BLOCK_SIZE."
)
assert "model" in globals() and "device" in globals(), (
    "The model is not loaded in this runtime. "
    "Complete Step 4 successfully before running Step 7."
)

work_dir = Path("/content/pointnet_classroom")
project = work_dir / "Pointnet_Pointnet2_pytorch"
data_root = work_dir / "student_data"
room_name = "Area_5_office_1.npy"
result_dir = work_dir / "results"

assert (data_root / room_name).is_file(), (
    "Room data not found. Complete Step 2 first."
)

if str(project) not in sys.path:
    sys.path.insert(0, str(project))

from data_utils.S3DISDataLoader import ScannetDatasetWholeScene

np.random.seed(SEED)
torch.manual_seed(SEED)

print("Step 7: Experiment with Block Size and Stride")
print(f"Device: {device}")
print(f"BLOCK_SIZE: {BLOCK_SIZE}")
print(f"STRIDE: {STRIDE}")
print(f"Points per block: {POINTS_PER_BLOCK}")
print("Voting passes: 1")
print("The entire room, including the ceiling, is used.")

# ------------------------------------------------------------
# 2. Load the room and generate blocks.
# ------------------------------------------------------------
print("\nLoading the room...", flush=True)

# Some classes may be absent from this room.
# Class-frequency weights are not used for this inference task.
with np.errstate(divide="ignore", invalid="ignore"):
    experiment_dataset = ScannetDatasetWholeScene(
        root=str(data_root) + "/",
        split="test",
        test_area=5,
        block_points=POINTS_PER_BLOCK,
        block_size=BLOCK_SIZE,
        stride=STRIDE
    )

experiment_dataset.labelweights = np.ones(
    NUM_CLASSES, dtype=np.float32
)

assert room_name in experiment_dataset.file_list, (
    "The target room was not found."
)

room_index = experiment_dataset.file_list.index(room_name)

print("Preparing room blocks. Please wait...", flush=True)

blocks, _, _, point_indices = experiment_dataset[room_index]

assert len(blocks) > 0, "No room blocks were generated."
assert np.isfinite(blocks).all(), (
    "The model input contains invalid numeric values."
)

xyzrgb = experiment_dataset.scene_points_list[room_index]
truth = (
    experiment_dataset.semantic_labels_list[room_index]
    .astype(np.int64)
)

assert ((truth >= 0) & (truth < NUM_CLASSES)).all(), (
    "Ground-truth labels are outside the valid class range."
)

num_blocks = len(blocks)

print(f"Room: {room_name}")
print(f"Original points: {len(truth):,}")
print(f"Blocks: {num_blocks:,}")

# ------------------------------------------------------------
# 3. Run inference and combine predictions by majority voting.
# ------------------------------------------------------------
votes = np.zeros(
    (len(truth), NUM_CLASSES), dtype=np.int32
)

model.eval()

# Only point features enter the model, not ground-truth labels.
with torch.inference_mode():
    for start in tqdm(
        range(0, num_blocks, BATCH_SIZE),
        desc="Predicting"
    ):
        end = min(start + BATCH_SIZE, num_blocks)

        inputs = torch.from_numpy(
            blocks[start:end].astype(np.float32)
        ).transpose(1, 2).contiguous().to(device)

        scores, _ = model(inputs)
        batch_predictions = scores.argmax(dim=2).cpu().numpy()

        original_indices = point_indices[start:end].astype(
            np.int64
        )

        np.add.at(
            votes,
            (
                original_indices.ravel(),
                batch_predictions.ravel()
            ),
            1
        )

covered = votes.sum(axis=1) > 0
assert covered.any(), "No points received predictions."

prediction = np.full(len(truth), -1, dtype=np.int64)
prediction[covered] = votes[covered].argmax(axis=1)

# Release large temporary arrays.
del blocks, point_indices, votes, inputs, scores

# ------------------------------------------------------------
# 4. Define the evaluation function.
# ------------------------------------------------------------
def evaluate_segmentation(y_true, y_pred, num_classes):
    """Calculate confusion matrix, overall accuracy, and IoU."""
    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)

    assert y_true.shape == y_pred.shape
    assert y_true.size > 0
    assert ((y_true >= 0) & (y_true < num_classes)).all()
    assert ((y_pred >= 0) & (y_pred < num_classes)).all()

    # Rows = ground-truth classes; columns = predicted classes.
    confusion = np.bincount(
        y_true.ravel() * num_classes + y_pred.ravel(),
        minlength=num_classes * num_classes
    ).reshape(num_classes, num_classes)

    true_positive = np.diag(confusion)
    true_count = confusion.sum(axis=1)
    predicted_count = confusion.sum(axis=0)

    # Union = TP + FP + FN.
    union = true_count + predicted_count - true_positive

    class_iou = np.full(num_classes, np.nan, dtype=np.float64)
    np.divide(
        true_positive,
        union,
        out=class_iou,
        where=union > 0
    )

    # Exclude classes absent from both truth and predictions.
    valid_classes = union > 0
    miou = class_iou[valid_classes].mean()
    overall_accuracy = true_positive.sum() / y_true.size

    return {
        "confusion": confusion,
        "true_count": true_count,
        "predicted_count": predicted_count,
        "true_positive": true_positive,
        "class_iou": class_iou,
        "valid_classes": valid_classes,
        "miou": miou,
        "overall_accuracy": overall_accuracy
    }


# Evaluate all covered room points, not a sampled subset.
metrics = evaluate_segmentation(
    truth[covered],
    prediction[covered],
    NUM_CLASSES
)

# ------------------------------------------------------------
# 5. Print overall metrics and per-class IoU.
# ------------------------------------------------------------
print("\n" + "=" * 66)
print("STEP 7 RESULTS")
print("=" * 66)

print(f"BLOCK_SIZE: {BLOCK_SIZE}")
print(f"STRIDE: {STRIDE}")
print(f"Blocks: {num_blocks:,}")
print(f"Total room points: {len(truth):,}")
print(f"Evaluated points: {covered.sum():,}")
print(f"Unpredicted points: {(~covered).sum():,}")
print(f"Prediction coverage: {covered.mean():.2%}")

print(
    f"\nOverall point accuracy: "
    f"{metrics['overall_accuracy']:.2%}"
)
print(f"Room mIoU: {metrics['miou']:.2%}")
print(
    f"Classes included in mIoU: "
    f"{metrics['valid_classes'].sum()}/{NUM_CLASSES}"
)

print("\nPer-class IoU")
header = (
    f"{'Class':<12}"
    f"{'True points':>14}"
    f"{'Predicted':>14}"
    f"{'Correct':>14}"
    f"{'IoU':>12}"
)
print(header)
print("-" * len(header))

for i, name in enumerate(CLASS_NAMES):
    iou_value = metrics["class_iou"][i]
    iou_text = (
        "N/A" if np.isnan(iou_value) else f"{iou_value:.2%}"
    )

    print(
        f"{name:<12}"
        f"{int(metrics['true_count'][i]):>14,}"
        f"{int(metrics['predicted_count'][i]):>14,}"
        f"{int(metrics['true_positive'][i]):>14,}"
        f"{iou_text:>12}"
    )

print("-" * len(header))
print(f"Room mIoU: {metrics['miou']:.2%}")

# ------------------------------------------------------------
# 6. Compare with Step 5 on commonly covered points.
# ------------------------------------------------------------
baseline_file = result_dir / "Area_5_office_1_prediction.npz"

if baseline_file.is_file():
    with np.load(baseline_file, allow_pickle=False) as baseline:
        same_room = (
            np.array_equal(baseline["xyzrgb"], xyzrgb)
            and np.array_equal(baseline["ground_truth"], truth)
        )

        if same_room:
            baseline_prediction = baseline["prediction"].astype(
                np.int64
            )
            baseline_covered = baseline["covered"].astype(bool)

    if same_room:
        common = covered & baseline_covered

        if common.any():
            baseline_metrics = evaluate_segmentation(
                truth[common],
                baseline_prediction[common],
                NUM_CLASSES
            )
            experiment_metrics = evaluate_segmentation(
                truth[common],
                prediction[common],
                NUM_CLASSES
            )

            print("\nComparison with Step 5")
            print(
                "Both runs are evaluated on the same "
                f"{common.sum():,} points."
            )

            print(
                f"\n{'Metric':<24}"
                f"{'Step 5':>12}"
                f"{'Step 7':>12}"
            )
            print("-" * 48)

            print(
                f"{'Overall point accuracy':<24}"
                f"{baseline_metrics['overall_accuracy']:>12.2%}"
                f"{experiment_metrics['overall_accuracy']:>12.2%}"
            )
            print(
                f"{'Room mIoU':<24}"
                f"{baseline_metrics['miou']:>12.2%}"
                f"{experiment_metrics['miou']:>12.2%}"
            )

            print(
                "\nPoint accuracy change: "
                f"{100 * (experiment_metrics['overall_accuracy'] - baseline_metrics['overall_accuracy']):+.2f}"
                " percentage points"
            )
            print(
                "Room mIoU change: "
                f"{100 * (experiment_metrics['miou'] - baseline_metrics['miou']):+.2f}"
                " percentage points"
            )

            # Explicitly report which classes contribute to each mean.
            for label, run_metrics in [
                ("Step 5", baseline_metrics),
                ("Step 7", experiment_metrics)
            ]:
                included = [
                    name
                    for name, valid in zip(
                        CLASS_NAMES, run_metrics["valid_classes"]
                    )
                    if valid
                ]
                print(
                    f"{label} mIoU classes: "
                    + ", ".join(included)
                )

            print(
                "Each mIoU excludes classes absent from both "
                "truth and that run's predictions."
            )
        else:
            print("\nNo common covered points. Comparison skipped.")
    else:
        print(
            "\nRoom data differs from Step 5. "
            "Baseline comparison skipped."
        )
else:
    print("\nStep 5 results not found. Baseline comparison skipped.")

# ------------------------------------------------------------
# 7. Save predictions and per-class metrics.
# ------------------------------------------------------------
result_dir.mkdir(parents=True, exist_ok=True)

experiment_file = result_dir / (
    f"Area_5_office_1_step7"
    f"_block{BLOCK_SIZE:g}_stride{STRIDE:g}.npz"
)

np.savez_compressed(
    experiment_file,
    xyzrgb=xyzrgb,
    ground_truth=truth,
    prediction=prediction,
    covered=covered,
    block_size=np.array(BLOCK_SIZE),
    stride=np.array(STRIDE),
    seed=np.array(SEED),
    num_blocks=np.array(num_blocks),
    class_names=np.array(CLASS_NAMES),
    class_iou=metrics["class_iou"],
    miou=np.array(metrics["miou"]),
    overall_accuracy=np.array(metrics["overall_accuracy"]),
    confusion_matrix=metrics["confusion"]
)

metrics_file = experiment_file.with_name(
    experiment_file.stem + "_class_iou.csv"
)

with metrics_file.open(
    "w", newline="", encoding="utf-8"
) as file:
    writer = csv.writer(file)
    writer.writerow([
        "class",
        "true_points",
        "predicted_points",
        "correct_points",
        "iou_percent"
    ])

    for i, name in enumerate(CLASS_NAMES):
        value = metrics["class_iou"][i]

        writer.writerow([
            name,
            int(metrics["true_count"][i]),
            int(metrics["predicted_count"][i]),
            int(metrics["true_positive"][i]),
            "" if np.isnan(value) else round(float(value) * 100, 4)
        ])

print("\nMetric notes:")
print("IoU = TP / (TP + FP + FN).")
print("Room mIoU is the unweighted mean of valid class IoUs.")
print("N/A means a class is absent from both truth and predictions.")
print("N/A classes are excluded from mIoU.")
print("A class with false positives but no true points has IoU = 0.")
print("All metrics use covered room points, including the ceiling.")
print("These are single-room metrics, not full-dataset scores.")